# From Qiskit circuit to Qubex pulses: an end-to-end visual tour

This tutorial walks the full provider pipeline with a **real
`qubex.Experiment`** (in mock mode — no hardware is contacted) and
visualizes every stage:

1. **Device topology** — generate Device Gateway topology metadata from the calibration data with `build_device_topology` and render it
2. **Backend** — construct a `qubex.Experiment` from the bundled offline configuration ([qubex-config/](qubex-config/)) and build a `QubexBackend` from the generated topology
3. **Circuit** — build a Qiskit circuit and transpile it with two scheduling methods (ALAP vs ASAP)
4. **Pulses** — compile each scheduled circuit to a Qubex `PulseSchedule` with `backend.validate(...)` (no execution)
5. **Visualize** — Gantt-style timelines (`build_pulse_schedule_timeline_figure`), Qubex's own waveform plot, and a numerical diff (`diff_pulse_schedules`)

Requirements: the `qubex` and `plot` extras. The `qubex` extra installs
Qubex together with its `qxdriver-quel1` backend package, which
`qubex.Experiment` imports even in mock mode:

```bash
pip install "qiskit-qubex-provider[qubex,plot] @ git+https://github.com/orangekame3/qiskit-qubex-provider.git"
```

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qubex import Experiment

from qiskit_qubex_provider import (
    QubexProvider,
    build_device_topology,
    build_device_topology_figure,
    build_pulse_schedule_timeline_figure,
    diff_pulse_schedules,
    extract_pulse_timeline,
    summarize_pulse_schedule,
    write_pulse_schedule_timeline_image,
)

## 1. Device topology — generated from the calibration data

`build_device_topology` turns Qubex calibration files into Device
Gateway `device-topology.json` metadata: qubits come from the
calibration note, couplings from its cross-resonance entries, and
fidelities / T1 / T2 from the metric files in the params directory.
Here we feed it the bundled [qubex-config/](qubex-config/) data —
representative demo values modeled on a real device calibration (see
[qubex-config/README.md](qubex-config/README.md) for where the numbers
come from).

The committed [device-topology.json](device-topology.json) /
[device-topology.svg](device-topology.svg) are the same output, produced
by the CLI:

```bash
qiskit-qubex-device-topology \
  --calib-note qubex-config/calibration/calib_note.json \
  --params-dir qubex-config/params \
  --name 4Q-DEMO --device-id 4Q-DEMO
```

In [2]:
topology = build_device_topology(
    calib_note_path="qubex-config/calibration/calib_note.json",
    params_dir="qubex-config/params",
    name="4Q-DEMO",
    device_id="4Q-DEMO",
)
build_device_topology_figure(topology).show()

## 2. A real `qubex.Experiment`, offline

[qubex-config/](qubex-config/) bundles everything a `qubex.Experiment`
needs to come up without hardware: a chip/system/box/wiring catalog, the
per-qubit frequency, amplitude, fidelity, and coherence parameters, and a
calibration note with single-qubit (`Drag`) and cross-resonance
calibration — representative values modeled on a real device
calibration. See
[qubex-config/README.md](qubex-config/README.md) for what each file is.

`mock_mode=True` mocks the control electronics, and
`calibration_valid_days` is raised so the committed calibration is not
discarded as stale. On a real setup you would point `config_dir`,
`params_dir`, and `calib_note_path` at your lab's configuration instead —
nothing else changes.

In [3]:
exp = Experiment(
    system_id="4Q-DEMO-SYS",
    muxes=[0],  # Q00-Q03
    config_dir="qubex-config/config",
    params_dir="qubex-config/params",
    calib_note_path="qubex-config/calibration/calib_note.json",
    calibration_valid_days=10000,  # committed example calibration; ignore staleness
    mock_mode=True,
)
exp.qubit_labels

[qubex.experiment.experiment_context] WARNING: Skew file not found: /Users/orangekame3/src/github.com/orangekame3/qiskit-qubex-provider/examples/qubex-config/config/skew.yaml


date: 2026-06-12 20:21:35


python: 3.13.11


qubex: 1.5.0b4


env: /Users/orangekame3/.cache/uv/builds-v0/.tmpnoVZPT


...

chip: 16Q-DEMO (16-qubit demo chip)


qubits: ['Q00', 'Q01', 'Q02', 'Q03']


muxes: ['MUX0']


boxes: ['DEMO2', 'DEMO1']


['Q00', 'Q01', 'Q02', 'Q03']

The calibrated pulse API now works exactly as on hardware — `x90` returns
the calibrated `Drag` pulse, `cx` builds the echoed cross-resonance
schedule from the stored CR parameters:

In [4]:
x90 = exp.x90("Q00")
print("x90(Q00):", type(x90).__name__, x90.duration, "ns")
print("cx(Q00, Q01) channels:", exp.pulse.cx("Q00", "Q01").labels)

x90(Q00): Drag 16.0 ns
cx(Q00, Q01) channels: ['Q00', 'Q00-Q01', 'Q01']


Build the backend from the experiment and the topology. The topology
supplies the coupling map and basis gates; the experiment supplies the
calibrated gate durations:

In [5]:
provider = QubexProvider.from_experiment(exp, device_topology=topology)
backend = provider.get_backend()
print("qubits:", backend.target.num_qubits)
print("basis:", sorted(backend.target.operation_names))

qubits: 4
basis: ['cx', 'cz', 'delay', 'ecr', 'h', 'id', 'measure', 'reset', 'rz', 's', 'sdg', 'sx', 'sxdg', 'x', 'y']


## 3. Build a circuit and transpile with two scheduling methods

The circuit runs two CX branches in parallel: a longer `h`+`x`+`rz`+`cx`
chain on `Q00`–`Q01` and an `h`+`cx` chain on `Q02`–`Q03`. The
`Q02`–`Q03` branch uses the slower CR pulse, so it sets the critical
path — which gives the scheduler a real choice for the `Q00`–`Q01`
branch: ALAP slides it late (everything aligned to the end), ASAP fires
it at *t* = 0. The `rz` becomes a `VirtualZ` (and, because a
cross-resonance channel drives in the target qubit's frame, it is
mirrored onto the `Q00-Q01` channel).

In [6]:
qc = QuantumCircuit(4, 4)
qc.h(0)
qc.x(1)
qc.rz(np.pi / 2, 1)
qc.cx(0, 1)
qc.h(2)
qc.cx(2, 3)
qc.measure(range(4), range(4))
qc.draw()

┌───┐                   ┌─┐   
q_0: ┤ H ├─────────────■─────┤M├───
     ├───┤┌─────────┐┌─┴─┐   └╥┘┌─┐
q_1: ┤ X ├┤ Rz(π/2) ├┤ X ├────╫─┤M├
     ├───┤└─────────┘└┬─┬┘    ║ └╥┘
q_2: ┤ H ├─────■──────┤M├─────╫──╫─
     └───┘   ┌─┴─┐    └╥┘ ┌─┐ ║  ║ 
q_3: ────────┤ X ├─────╫──┤M├─╫──╫─
             └───┘     ║  └╥┘ ║  ║ 
c: 4/══════════════════╩═══╩══╩══╩═
                       2   3  0  1

In [7]:
alap = transpile(qc, backend, scheduling_method="alap", optimization_level=1)
asap = transpile(qc, backend, scheduling_method="asap", optimization_level=1)

## 4. Compile to Qubex pulse schedules

`backend.validate(...)` converts a scheduled circuit into the Qubex
`PulseSchedule` that would run on hardware — without executing anything.

In [8]:
schedule_alap = backend.validate(alap)[0]
schedule_asap = backend.validate(asap)[0]

## 5. Visualize

### Gantt-style timeline (Qiskit `timeline_drawer`-like)

One horizontal lane per channel — drive (`Q00`…), cross-resonance
(`Q00-Q01`), and readout (`RQ00`…) — with a labeled box per pulse (colored
by waveform name) and tall labeled ticks for virtual-Z phase shifts. The
boxes are the calibrated waveforms: `Drag` for single-qubit gates,
`FlatTop` for the CR drive, cancel tones, and readout. One figure per
schedule: render both and compare side by side.

In [9]:
build_pulse_schedule_timeline_figure(schedule_alap, title="scheduling_method=alap").show()

In [10]:
build_pulse_schedule_timeline_figure(schedule_asap, title="scheduling_method=asap").show()

Both cross-resonance lanes (`Q00-Q01` and `Q03-Q02`) are now in play.
The `Q02`–`Q03` branch is the critical path, so it is identical in both
figures — while the whole `Q00`–`Q01` branch (drive pulses, its CR lane,
and the `RQ00`/`RQ01` readouts) slides between "as late as possible"
and *t* = 0. ASAP also starts the `Q00`/`Q01` readouts as soon as that
branch finishes, while ALAP aligns all four readouts at the end.

To share a figure (e.g. in a PR description), write it to a file. `.html`
needs only Plotly; `.png`/`.svg` additionally need Kaleido.

In [11]:
write_pulse_schedule_timeline_image(
    schedule_alap, "timeline-alap.html", title="scheduling_method=alap"
)

The timeline data behind the figure is also available programmatically:

In [12]:
extract_pulse_timeline(schedule_alap)["Q00"]

[{'kind': 'phase',
  'name': 'VirtualZ',
  'start_ns': 80.0,
  'duration_ns': 0.0,
  'theta': -3.141592653589793},
 {'kind': 'pulse', 'name': 'Drag', 'start_ns': 80.0, 'duration_ns': 16.0},
 {'kind': 'pulse', 'name': 'Drag', 'start_ns': 368.0, 'duration_ns': 24.0},
 {'kind': 'pulse', 'name': 'Drag', 'start_ns': 664.0, 'duration_ns': 24.0},
 {'kind': 'phase',
  'name': 'VirtualZ',
  'start_ns': 688.0,
  'duration_ns': 0.0,
  'theta': 1.5707963267948966}]

### Waveform detail via Qubex

For the actual waveform envelopes (X/Y components and frame phase per
channel), call Qubex's own plot on the schedule — no provider API needed:

In [13]:
schedule_alap.plot(title="scheduling_method=alap (waveform detail)")

### Numerical verdict

`diff_pulse_schedules` compares two schedules sample by sample. The
verdict matches the figures: every channel of the `Q00`–`Q01` branch
changed (drive, CR, and readout), while the critical-path `Q02`–`Q03`
branch is identical.

In [14]:
diff = diff_pulse_schedules(schedule_alap, schedule_asap)
print("equal:", diff["equal"])
diff["channels"]

equal: False


{'Q00': {'status': 'changed', 'max_abs_diff': 0.4206603110370107},
 'Q01': {'status': 'changed', 'max_abs_diff': 0.27229418834906516},
 'Q02': {'status': 'equal', 'max_abs_diff': 0.0},
 'Q03': {'status': 'equal', 'max_abs_diff': 0.0},
 'Q00-Q01': {'status': 'changed', 'max_abs_diff': 1.5624000477642606},
 'Q03-Q02': {'status': 'equal', 'max_abs_diff': 0.0},
 'RQ00': {'status': 'changed', 'max_abs_diff': 0.119815},
 'RQ01': {'status': 'changed', 'max_abs_diff': 0.121878},
 'RQ02': {'status': 'equal', 'max_abs_diff': 0.0},
 'RQ03': {'status': 'equal', 'max_abs_diff': 0.0}}

`summarize_pulse_schedule` gives the per-channel timing facts behind that
verdict (total duration and first/last non-blank sample boundaries):

In [15]:
summarize_pulse_schedule(schedule_alap)

{'Q00': {'duration_ns': 1248.0,
  'active_start_ns': 80.0,
  'active_end_ns': 688.0,
  'n_samples': 624},
 'Q01': {'duration_ns': 1248.0,
  'active_start_ns': 72.0,
  'active_end_ns': 704.0,
  'n_samples': 624},
 'Q02': {'duration_ns': 1248.0,
  'active_start_ns': 16.0,
  'active_end_ns': 704.0,
  'n_samples': 624},
 'Q03': {'duration_ns': 1248.0,
  'active_start_ns': 0.0,
  'active_end_ns': 704.0,
  'n_samples': 624},
 'Q00-Q01': {'duration_ns': 1248.0,
  'active_start_ns': 96.0,
  'active_end_ns': 664.0,
  'n_samples': 624},
 'Q03-Q02': {'duration_ns': 1248.0,
  'active_start_ns': 16.0,
  'active_end_ns': 648.0,
  'n_samples': 624},
 'RQ00': {'duration_ns': 1248.0,
  'active_start_ns': 736.0,
  'active_end_ns': 1120.0,
  'n_samples': 624},
 'RQ01': {'duration_ns': 1248.0,
  'active_start_ns': 736.0,
  'active_end_ns': 1120.0,
  'n_samples': 624},
 'RQ02': {'duration_ns': 1248.0,
  'active_start_ns': 736.0,
  'active_end_ns': 1120.0,
  'n_samples': 624},
 'RQ03': {'duration_ns': 1248.

## Wrap-up

On a real setup, point `config_dir` / `params_dir` / `calib_note_path` at
your lab's Qubex configuration and drop `mock_mode` — the pipeline is
identical: topology → backend → transpile → `validate` → visualize. The
same schedules then run for real via `backend.run(...)`.

Related views:

- Waveform detail (X/Y, phase): Qubex's own `schedule.plot()` (shown above)
- Gate-level timeline of the scheduled circuit:
  `qiskit.visualization.timeline_drawer(alap)` (requires matplotlib)

See also: [docs/pulse-schedule-visualization.md](../docs/pulse-schedule-visualization.md)
and [docs/device-topology.md](../docs/device-topology.md).